# 02 - Validacao PyTorch

Recria a MLP da Etapa 1 em PyTorch para comparar curvas no mesmo problema 28x28 e, separadamente, valida o pipeline PathMNIST 224x224 que passa a ser usado nas etapas seguintes.

In [ ]:
import importlib.util
import os
import random
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

def ensure_package(package, import_name=None):
    import_name = import_name or package.split('==')[0].replace('-', '_')
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

if IN_COLAB:
    ensure_package('medmnist==3.0.2', 'medmnist')
    ensure_package('pandas', 'pandas')
    ensure_package('matplotlib', 'matplotlib')

project_candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AP2_IA'), Path('/content/ap2-ia'), Path('/content/drive/MyDrive/AP2_IA')]
PROJECT_ROOT = next((path for path in project_candidates if (path / 'src' / 'train.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Nao encontrei a raiz do projeto. Execute o notebook dentro da pasta do repositorio.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from medmnist import PathMNIST
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from data.dataset import get_loaders
from utils import device, save_json, set_seed

SEED = 42
set_seed(SEED)
DEVICE = device()
ARTIFACT_DIR = PROJECT_ROOT / 'experiments' / 'stage02_pytorch_validation'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('Projeto:', PROJECT_ROOT)
print('Dispositivo:', DEVICE)

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 9),
        )

    def forward(self, x):
        return self.net(x)

train_ds = PathMNIST(split='train', size=28, download=True)
val_ds = PathMNIST(split='val', size=28, download=True)

def prepare_torch_split(dataset):
    x_np = dataset.imgs.astype('float32') / 255.0
    x_np = x_np.mean(axis=-1).reshape(len(x_np), -1)
    y_np = dataset.labels.reshape(-1).astype('int64')
    return torch.tensor(x_np, dtype=torch.float32), torch.tensor(y_np, dtype=torch.long)

x_train, y_train = prepare_torch_split(train_ds)
x_val, y_val = prepare_torch_split(val_ds)
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)

model = TorchMLP().to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
criterion = nn.CrossEntropyLoss()
history_torch = []

for epoch in range(1, 21):
    losses = []
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        train_pred = model(x_train[:5000].to(DEVICE)).argmax(1).cpu()
        val_pred = model(x_val.to(DEVICE)).argmax(1).cpu()
    row = {
        'epoch': epoch,
        'loss_train': float(sum(losses) / len(losses)),
        'acc_train_sample': float((train_pred == y_train[:5000]).float().mean().item()),
        'acc_val': float((val_pred == y_val).float().mean().item()),
    }
    history_torch.append(row)
    print(row)

torch_df = pd.DataFrame(history_torch)
torch_df.to_csv(ARTIFACT_DIR / 'torch_mlp_history.csv', index=False)
torch_df.tail()

In [ ]:
numpy_path = PROJECT_ROOT / 'experiments' / 'stage01_numpy' / 'numpy_mlp_history.csv'
comparison = {'torch_final_val_acc': float(torch_df['acc_val'].iloc[-1])}
fig, ax = plt.subplots(figsize=(8, 4))
torch_df.plot(x='epoch', y='acc_val', ax=ax, label='PyTorch MLP val')
if numpy_path.exists():
    numpy_df = pd.read_csv(numpy_path)
    numpy_df.plot(x='epoch', y='acc_val', ax=ax, label='NumPy MLP val')
    diff_pp = abs(torch_df['acc_val'].iloc[-1] - numpy_df['acc_val'].iloc[-1]) * 100
    comparison.update({'numpy_final_val_acc': float(numpy_df['acc_val'].iloc[-1]), 'final_accuracy_diff_pp': float(diff_pp), 'criterion_met_diff_le_2pp': bool(diff_pp <= 2.0)})
else:
    comparison['note'] = 'Execute o notebook 01 antes para comparar com NumPy.'
ax.set_title('Validacao NumPy vs PyTorch no mesmo problema 28x28')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'stage02_numpy_vs_pytorch.png', dpi=160)
save_json(ARTIFACT_DIR / 'numpy_vs_pytorch_comparison.json', comparison)
comparison

In [ ]:
loaders_224 = get_loaders(batch_size=4, image_size=224, source_size=28, num_workers=0)
images, labels = next(iter(loaders_224['train']))
pipeline_info = {
    'image_shape': list(images.shape),
    'label_shape': list(labels.shape),
    'batch_size': int(images.shape[0]),
    'image_size': 224,
    'uses_torchvision_transforms': True,
    'normalized_with_imagenet_stats': True,
    'source_size': 28,
    'memory_safe_resize_to_224': True,
}
save_json(ARTIFACT_DIR / 'pathmnist_224_pipeline_check.json', pipeline_info)
pipeline_info

A comparacao NumPy/PyTorch usa 28x28 para manter a mesma entrada da Etapa 1. O arquivo `pathmnist_224_pipeline_check.json` documenta que, a partir desta etapa, o pipeline 224x224 esta pronto para CNNs e backbones pre-treinados.